# NDP Affinities API Tutorial

**API Version: 0.1.1**

This notebook demonstrates how to use all endpoints of the NDP Affinities API.

## Setup

First, let's set up our base URL and import the required libraries.

In [ ]:
import requests
import json

# Change this to your API URL
BASE_URL = "http://localhost:8000"

def pretty_print(data):
    """Pretty print JSON data"""
    print(json.dumps(data, indent=2))

## 1. Health Check

Verify the API is running.

In [ ]:
response = requests.get(f"{BASE_URL}/health")
pretty_print(response.json())

## 2. Endpoints (`/ep`)

Manage data endpoints (e.g., CKAN instances, APIs, data sources).

### 2.1 Create an Endpoint

In [ ]:
endpoint_data = {
    "kind": "ckan",
    "url": "https://demo.ckan.org",
    "metadata": {
        "name": "CKAN Demo",
        "description": "Official CKAN demonstration site"
    }
}

response = requests.post(f"{BASE_URL}/ep", json=endpoint_data)
endpoint = response.json()
pretty_print(endpoint)

# Save the UID for later use
endpoint_uid = endpoint["uid"]

### 2.2 List All Endpoints

In [ ]:
response = requests.get(f"{BASE_URL}/ep")
endpoints = response.json()
print(f"Total endpoints: {len(endpoints)}")
pretty_print(endpoints[:2])  # Show first 2

### 2.3 Get a Specific Endpoint

In [ ]:
response = requests.get(f"{BASE_URL}/ep/{endpoint_uid}")
pretty_print(response.json())

### 2.4 Update an Endpoint

In [ ]:
update_data = {
    "metadata": {
        "name": "CKAN Demo (Updated)",
        "description": "Official CKAN demonstration site",
        "updated": True
    }
}

response = requests.patch(f"{BASE_URL}/ep/{endpoint_uid}", json=update_data)
pretty_print(response.json())

### 2.5 Delete an Endpoint

In [ ]:
response = requests.delete(f"{BASE_URL}/ep/{endpoint_uid}")
print(f"Status: {response.status_code}")

## 3. Datasets (`/datasets`)

Manage dataset records.

### 3.1 Create a Dataset

In [ ]:
dataset_data = {
    "title": "Climate Data 2024",
    "metadata": {
        "description": "Global climate measurements",
        "format": "CSV",
        "size": "1.2GB"
    }
}

response = requests.post(f"{BASE_URL}/datasets", json=dataset_data)
dataset = response.json()
pretty_print(dataset)

dataset_uid = dataset["uid"]

### 3.2 List All Datasets

In [ ]:
response = requests.get(f"{BASE_URL}/datasets")
datasets = response.json()
print(f"Total datasets: {len(datasets)}")
pretty_print(datasets[:2])

### 3.3 Get a Specific Dataset

In [ ]:
response = requests.get(f"{BASE_URL}/datasets/{dataset_uid}")
pretty_print(response.json())

## 4. Services (`/services`)

Manage service records (APIs, processing services, etc.).

### 4.1 Create a Service

In [ ]:
service_data = {
    "type": "processing",
    "openapi_url": "https://api.example.com/openapi.json",
    "version": "1.0.0",
    "metadata": {
        "name": "Data Processing Service",
        "description": "Transforms and aggregates data"
    }
}

response = requests.post(f"{BASE_URL}/services", json=service_data)
service = response.json()
pretty_print(service)

service_uid = service["uid"]

### 4.2 List All Services

In [ ]:
response = requests.get(f"{BASE_URL}/services")
services = response.json()
print(f"Total services: {len(services)}")
pretty_print(services[:2])

## 5. Relationships

Connect datasets with endpoints and services.

### 5.1 Link Dataset to Endpoint (`/dataset-endpoints`)

In [ ]:
# First, create a new endpoint for this example
ep_response = requests.post(f"{BASE_URL}/ep", json={"kind": "api", "url": "https://api.example.com"})
new_endpoint_uid = ep_response.json()["uid"]

# Link dataset to endpoint
link_data = {
    "dataset_uid": dataset_uid,
    "endpoint_uid": new_endpoint_uid,
    "role": "source",
    "attrs": {"priority": 1}
}

response = requests.post(f"{BASE_URL}/dataset-endpoints", json=link_data)
pretty_print(response.json())

### 5.2 Link Dataset to Service (`/dataset-services`)

In [ ]:
link_data = {
    "dataset_uid": dataset_uid,
    "service_uid": service_uid,
    "role": "processor",
    "attrs": {"schedule": "daily"}
}

response = requests.post(f"{BASE_URL}/dataset-services", json=link_data)
pretty_print(response.json())

### 5.3 Link Service to Endpoint (`/service-endpoints`)

In [ ]:
link_data = {
    "service_uid": service_uid,
    "endpoint_uid": new_endpoint_uid,
    "role": "provider"
}

response = requests.post(f"{BASE_URL}/service-endpoints", json=link_data)
pretty_print(response.json())

## 6. Affinities (`/affinities`)

Create affinity triples (dataset + endpoints + services combinations).

### 6.1 Create an Affinity

In [ ]:
affinity_data = {
    "dataset_uid": dataset_uid,
    "endpoint_uids": [new_endpoint_uid],
    "service_uids": [service_uid],
    "attrs": {"confidence": 0.95}
}

response = requests.post(f"{BASE_URL}/affinities", json=affinity_data)
affinity = response.json()
pretty_print(affinity)

affinity_uid = affinity["triple_uid"]

### 6.2 List All Affinities

In [ ]:
response = requests.get(f"{BASE_URL}/affinities")
affinities = response.json()
print(f"Total affinities: {len(affinities)}")
pretty_print(affinities[:2])

## 7. Linked Entities (`/linked`)

Query related entities for a given UID.

### 7.1 Get Linked Entities for a Single UID

In [ ]:
response = requests.get(f"{BASE_URL}/linked/{dataset_uid}")
pretty_print(response.json())

### 7.2 Batch Query for Multiple UIDs

In [ ]:
batch_data = {
    "uids": [dataset_uid, service_uid]
}

response = requests.post(f"{BASE_URL}/linked/batch", json=batch_data)
pretty_print(response.json())

## 8. Cleanup

Delete the resources created in this tutorial.

In [ ]:
# Delete in reverse order of dependencies
requests.delete(f"{BASE_URL}/affinities/{affinity_uid}")
requests.delete(f"{BASE_URL}/dataset-endpoints/{dataset_uid}/{new_endpoint_uid}")
requests.delete(f"{BASE_URL}/dataset-services/{dataset_uid}/{service_uid}")
requests.delete(f"{BASE_URL}/service-endpoints/{service_uid}/{new_endpoint_uid}")
requests.delete(f"{BASE_URL}/datasets/{dataset_uid}")
requests.delete(f"{BASE_URL}/services/{service_uid}")
requests.delete(f"{BASE_URL}/ep/{new_endpoint_uid}")

print("Cleanup complete!")

## Summary

This tutorial covered all main endpoints of the NDP Affinities API:

| Endpoint | Description |
|----------|-------------|
| `/health` | Health check |
| `/ep` | Manage endpoints |
| `/datasets` | Manage datasets |
| `/services` | Manage services |
| `/dataset-endpoints` | Link datasets to endpoints |
| `/dataset-services` | Link datasets to services |
| `/service-endpoints` | Link services to endpoints |
| `/affinities` | Manage affinity triples |
| `/linked` | Query related entities |

For more details, visit the Swagger UI at `/docs`.